In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee
import datetime

ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

# Brazil bounding box
bbox = ee.Geometry.BBox(-94.1875, -39.0208, 37.0625, 18.2292)

# CHIRPS dataset
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(bbox)

# Range of years - UPDATE THIS FOR NEW YEARS
years = list(range(2025, 2026))  # For 2025 data

print(f"Exporting CHIRPS data for Brazil: {years}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")

In [ ]:
# Loop through years and export as yearly multi-band images
for year in years:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    # Filter to that year's daily images
    year_coll = chirps.filterDate(start, end).select('precipitation')

    # Merge all days into a single multi-band image
    year_img = year_coll.toBands()

    # Add time metadata
    year_img = year_img.set('system:time_start', start.millis())

    # Export to Drive
    task = ee.batch.Export.image.toDrive(
        image=year_img.clip(bbox),
        description=f"CHIRPS_Daily_Brazil_{year}",
        folder="Brazil_CHIRPS_Daily",
        fileNamePrefix=f"CHIRPS_Daily_{year}",
        region=bbox,
        scale=5000,
        maxPixels=1e13
    )
    task.start()
    print(f"Exporting daily CHIRPS for {year} ...")

print("\nAll CHIRPS export tasks initiated!")
print("Check Google Earth Engine Tasks tab for progress.")